# Classification in Bagging
 In this Each subset is used to train a separate "base learner" (most commonly a Decision Tree). For classification, the final prediction is determined by majority voting among all the trained models.

In [1]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

In [2]:
X,y = make_classification(n_samples=10000, n_features=10, n_informative=3)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
X_train.shape,X_test.shape

((8000, 10), (2000, 10))

# Case 1. How a single Decision Tree is performing?

In [5]:
dt=DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)
y_pred=dt.predict(X_test)
print(' Decision Tree accuracy ',accuracy_score(y_test,y_pred))

 Decision Tree accuracy  0.8445


# Case 2. Bagging Using Decision Tree

In [6]:
bag_dt = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500, # no. of trees
    max_samples=0.25, # train_data(8000), 8000*0.25=2000 rows for every tree
    bootstrap=True, # sampling with replacement(means same row can come multiple times)
    random_state=42
)

In [7]:
bag_dt.fit(X_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, random_state=42)

In [8]:
y_pred = bag_dt.predict(X_test)

In [9]:
accuracy_score(y_test,y_pred)

0.8965

In [10]:
bag_dt.estimators_samples_[0:3] # first three estimators contains this sample dataset

[array([2523, 3113, 7114, ..., 4291, 4472, 3620]),
 array([4782,  663, 7155, ..., 5963,  495, 1767]),
 array([5462, 6574, 4896, ..., 3979, 7827,   37])]

In [11]:
bag_dt.estimators_samples_[0].shape,bag_dt.estimators_features_[0].shape

((2000,), (10,))

# Case 3. Bagging Using SVM

In [12]:
bag_svm = BaggingClassifier(
    estimator=SVC(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42
)

In [13]:
bag_svm.fit(X_train,y_train)
y_pred = bag_svm.predict(X_test)
print('Bag_SVM accuracy ',accuracy_score(y_test,y_pred))

Bag_SVM accuracy  0.892


# Case 4. Pasting
- Row Sampling without Replacement (no row can come multiple times)

In [14]:
bag_pasting = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False, # without replacement
    random_state=42,
    verbose=1,
    n_jobs=-1
)
bag_pasting.fit(X_train,y_train)
y_pred = bag_pasting.predict(X_test)
print('Bag_Pasting accuracy ',accuracy_score(y_test,y_pred))

[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:   18.9s finished
[Parallel(n_jobs=2)]: Using backend LokyBackend with 2 concurrent workers.


Bag_Pasting accuracy  0.898


[Parallel(n_jobs=2)]: Done   2 out of   2 | elapsed:    0.7s finished


# Case 5. Random Subspaces
- Feature Sampling or Column Sampling

In [15]:
bag_subspaces = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    bootstrap=False, # without replacement in rows
    max_features=0.5,
    bootstrap_features=True, # Sampling with replacement for columns
    random_state=42,
)

bag_subspaces.fit(X_train,y_train)
y_pred = bag_subspaces.predict(X_test)
print('Bag_Subspaces accuracy ',accuracy_score(y_test,y_pred))

Bag_Subspaces accuracy  0.89


In [16]:
bag_subspaces.estimators_samples_[0].shape

(8000,)

In [17]:
bag_subspaces.estimators_features_[0].shape # Total feature(Column=10) then, 10*0.5 = 5 column

(5,)

# Case 6. Random Patches
 - Both Row Sampling and Column Sampling

In [18]:
bag_patches = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.2,
    bootstrap=True, # Sampling with replacement in rows
    max_features=0.5,
    bootstrap_features=True, # Sampling with replacement for columns
    random_state=42,
)

bag_patches.fit(X_train,y_train)
y_pred = bag_patches.predict(X_test)
print('Bag_Patches accuracy ',accuracy_score(y_test,y_pred))

Bag_Patches accuracy  0.8985


In [19]:
bag_patches.estimators_samples_[0].shape, bag_patches.estimators_features_[0].shape

((1600,), (5,))

# Case 7. OOB Score(OUT OF BAG Score)

In [20]:
bag_oob = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True,
    random_state=42,
)

In [21]:
bag_oob.fit(X_train,y_train)

BaggingClassifier(estimator=DecisionTreeClassifier(), max_samples=0.25,
                  n_estimators=500, oob_score=True, random_state=42)

In [22]:
# 33% data is in oob, which we can use for validation in bagging
bag_oob.oob_score_

0.902375

In [23]:
y_pred = bag_oob.predict(X_test)
print('Bag_OOB accuracy ',accuracy_score(y_test,y_pred))

Bag_OOB accuracy  0.8965


# Bagging Tips
  - Bagging generally gives better results than Pasting
  - Good Results come around the 25% to 50% row sampling mark
  - Random patches and subspaces should be used while dealing with high dimensional data
  - To find the correct hyperparameter values we can do GridSearchCV/RandomSearchCV

# Applying GridSearchCV

In [26]:
from sklearn.model_selection import GridSearchCV

parameters = {
    'n_estimators': [10, 50],
    'max_samples': [0.5, 1.0],
    'bootstrap': [True],
    'max_features': [0.5, 1.0]
}

In [27]:
search = GridSearchCV(BaggingClassifier(), parameters, cv=5)
search.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=BaggingClassifier(),
             param_grid={'bootstrap': [True], 'max_features': [0.5, 1.0],
                         'max_samples': [0.5, 1.0], 'n_estimators': [10, 50]})

In [28]:
search.best_params_

{'bootstrap': True,
 'max_features': 1.0,
 'max_samples': 0.5,
 'n_estimators': 50}

In [29]:
search.best_score_

np.float64(0.899125)

# Apply Randomized Search CV

In [30]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import BaggingRegressor

random_search = RandomizedSearchCV(
    BaggingRegressor(),
    param_distributions=parameters,
    n_iter=10,   # only 10 random combinations
    cv=3,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


RandomizedSearchCV(cv=3, estimator=BaggingRegressor(), n_jobs=-1,
                   param_distributions={'bootstrap': [True],
                                        'max_features': [0.5, 1.0],
                                        'max_samples': [0.5, 1.0],
                                        'n_estimators': [10, 50]},
                   random_state=42)

In [31]:
random_search.best_params_

{'n_estimators': 50,
 'max_samples': 0.5,
 'max_features': 1.0,
 'bootstrap': True}

In [34]:
random_search.best_score_

np.float64(0.7006405529993506)